# AI Modified Workflow

For this workflow we use the following prompt to modify the `scale_model_size.py` notebook:

Structure this workflow to run twice once using SST version 15.0 and once using SST version 15.1. Present both results in a single gaph that I can use to compare the relative performance.

---

If I run "as is", in the "Download Containers" step I'll error out with:

```
Error: initializing source docker://ghcr.io/hpc-ai-adv-dev/sst-core:v15.0.0: reading manifest v15.0.0 in ghcr.io/hpc-ai-adv-dev/sst-core: manifest unknown
```

This is because it should be `sst-core:15.0.0` (that is no `v`).  It's easy enough to just rewrite the assignment to fix it.

I'd also have an issue later on that my output parsing logic isn't backwards compatible with v15.0.
It's not fair to expect that copilot could resolve this, so in the interest of evaluating it, to test it I modified it to do a comparison between v15.1.0 and v15.1.1.

In other words, I added the following to the end of the "global params" cell:

```
sst_container_uris = {
    '15.1.0': 'ghcr.io/hpc-ai-adv-dev/sst-core:15.1.0',
    '15.1.1': 'ghcr.io/hpc-ai-adv-dev/sst-core:15.1.1',
}
```

Anyway, if I resolve those issues, the next issue is that it only builds PHOLD for the first version. It helpefully has a comment explaining this:

```
# Build once using the first configured SST container (PHOLD build output is shared across runs).
```

But this is indeed could be a problem.

Funny enough for versions 15.1.0 and 15.1.1, it turns out they're ABI compatible and so nothing bad happened.  With other versions this could lead to linker errors.  So it would be good to inform copilot that we are comparing a benchmark across SST versions we need to build it in each of these versions.

Moving on, I'm actually able to produce the graph comparing PHOLD across the two versions.

Some additional thoughts:

- I like how copilot took the original `container_url` and expanded it to address multiple containers by turning it into a dictionary called `sst_container_uris`. The comment update was also good although it cut out some details I had in the original comment about how to view available containers on the container factory. I would like to have kept that comment.

- When comparing across versions that might mean passing in arguments with different names, etc.  I think in this case we would like to also allow the user to modify variables like `sst_args_template` across runs.

- Copilot also generalized the `container_name` variable but cut out my comment telling the user that they shouldn't modify it. Copilot replaced it with a comment that "[it is] Populated in the container download section", but I prefer my original comment that had all-capped text telling the user to "NOT MODIFY THIS LINE".

- I like that copilot put log files for each SST version into separate directories. It was also smart enough to name these directories after the version number (e.g. 15.0.0) but replaced the periods with underscores (e.g. 15.0.0 -> 15_0_0).

- It's interesting that copilot generated two separate csv files (one per version) and than used pandas to read in the CSVs and combine them into a single unified `results.csv` with the additional column for SST version.  When I manually did this workflow in the past what I did was produce multiple CSVs (one per each SST version) and then read them in individually later when I produced the plot. I'm not sure which approach I like more.  This approach does add additional complexity to the preprocessing step and I think we'd want to try and abstract some of this into the workflows module.

---
---
---

# Configuration

## Import workflows module

In [ ]:
from utils.workflows import *

## Global params

Users can modify these top-level parameters to alter the behavior of this workflow.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------------
# !!! DO NOT MODIFY THE CODE BELOW   !!!
# !!!  (Modify in the next section)  !!!
# ---------------------------------------------------------------------------------------------------------------------

# So that can you can maintain the defaults, we suggest you don't directly edit
# the parameters inline here but rather overwite values at the bottom of this
# cell.

# We'll store our containers and benchmark results under the specified directory
# (it will be created if it doesn't already exist).
import os
if 'user_customExperimentsDir' in globals():
    baseDir = f'{user_customExperimentsDir}/scale_model_size'
else:
    baseDir=f'{os.getenv("HOME")}/workflows/scale_model_size'

# Run using SST containers for each version we want to compare. The values in this
# dict are fully-qualified container URIs and the keys are display labels used in
# run directory names and plots.
sst_container_uris = {
    '15.0': 'ghcr.io/hpc-ai-adv-dev/sst-core:v15.0.0',
    '15.1': 'ghcr.io/hpc-ai-adv-dev/sst-core:v15.1.0',
}
container_names = {}  # Populated in the container download section.

# The benchmark will be cloned from the specified repository. We assume the
# benchmark itself is in the 'benchmarkPath' directory within the repos.  We
# assume building the benchmark is a matter of running 'make' in that directoy.
benchmarkRepos='https://github.com/hpc-ai-adv-dev/sst-benchmarks.git'
benchmarkPath='phold'

# Run the benchmark on a single node, increasing the numbers of components with each trial
num_comps_per_trial  = [1_000_000, 2_000_000, 3_000_000, 4_000_000, 5_000_000]

# This command will be run prior to launching a job. The command will be run
# from within the benchmark directory and execution occurs within the worklaunch
# loop so it may be parameterized by the trial parameters if needed.
prestart_cmd_template = ''

# Indicates what arguments should be passed to sst and the benchmark each run 
# Note: {width} and {height} will be replaced with the appropriate values for
# each run, based on the number of nodes and components per node
sst_args_template   = '--print-timing-info=3 --parallel-load=SINGLE ./phold_dist.py'
bmark_args_template = '--width {width} --height {height}'

# Additional arguments to pass when launching jobs with srun. For example the
# partition name or --qos=high for higher priority in the queue.
additional_srun_args = ''

# Several of the setup steps will avoid rerunning if they have previously been run. Append to this
# list to indicate when you want to force a step to be reproduced.
#
# VALID VALUES ARE:
#   'ALL'     
#   'DOWNLOAD_CONTAINERS' 
#   'DOWNLOAD_BENCHMARKS' 
#   'BUILD_BENCHMARKS'      Note: we always rerun make, if this is set we will also run 'make clean' before rebuilding
force = []

# ---------------------------------------------------------------------------------------------------------------------
# Overwite parameters below this line to customize the workflow: 
# ---------------------------------------------------------------------------------------------------------------------
sst_container_uris = {
    '15.1.0': 'ghcr.io/hpc-ai-adv-dev/sst-core:15.1.0',
    '15.1.1': 'ghcr.io/hpc-ai-adv-dev/sst-core:15.1.1',
}

num_comps_per_trial  = [1000, 2000, 3000, 4000, 5000]

## Environment

In [ ]:
set_workflow_log(f'{baseDir}/workflow.log')
run_cmd(f'e4s-cl profile edit --add-files {baseDir}')

## Download containers

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_CONTAINERS' in force

container_names = {}
for sst_version, container_uri in sst_container_uris.items():
    container_names[sst_version] = download_custom_container(container_uri, force=_force)

print('Downloaded/available containers:')
for sst_version, container_name in container_names.items():
    print(f'  SST {sst_version}: {container_name}')

## Download benchmarks

In [ ]:
_force = 'ALL' in force or 'DOWNLOAD_BENCHMARKS' in force

if not os.path.exists(f'benchmarks') or _force:
    run_cmd(f"git clone {benchmarkRepos} benchmarks")
    run_cmd(f"e4s-cl profile edit --add-files {baseDir}/benchmarks/{benchmarkPath}")
else:
    print(f"Benchmarks from {benchmarkRepos} have already been downloaded, skipping download.")

## Build benchmarks 

In [ ]:
_force = 'ALL' in force or 'BUILD_BENCHMARKS' in force

cd(f"{baseDir}/benchmarks/{benchmarkPath}")
run_cmd('touch sstsimulator.conf')
_cmd = 'make' if not _force else 'make clean; make'

# Build once using the first configured SST container (PHOLD build output is shared across runs).
build_container_name = container_names[next(iter(sst_container_uris))]
run_in_container(_cmd,
    f'{baseDir}/{build_container_name}',
    additional_apptainer_args=f'--bind sstsimulator.conf:{os.getenv("HOME")}/.sst/sstsimulator.conf')
cd(baseDir)

# Run

## Start jobs

In [ ]:
import math, shutil, os

runDisplay = SafeDisplay(display_handle = display('', display_id="run_disp"))

# Setup directory to store results in
run_root_dir = f'{baseDir}/runs/'
if os.path.exists(run_root_dir):
    shutil.rmtree(run_root_dir)
os.makedirs(run_root_dir, exist_ok=True)

cd(f"{baseDir}/benchmarks/{benchmarkPath}")

# Deploy jobs for each SST version and each trial size.
for sst_version, container_name in container_names.items():
    run_dir = f'{run_root_dir}/sst_{sst_version.replace(".", "_")}'
    os.makedirs(run_dir, exist_ok=True)

    for approx_size in num_comps_per_trial:
        width  = int(math.sqrt(approx_size))
        height = width
        size = width*height

        if prestart_cmd_template is not None and prestart_cmd_template != '':
            run_cmd(prestart_cmd_template.format(width=width, height=height, size=size))

        full_sst_args_template = f'{sst_args_template} -- {bmark_args_template}'
        sst_args = full_sst_args_template.format(width=width, height=height, size=size)

        launch_and_log_sst(
            image        = f'{baseDir}/{container_name}',
            srun_args    = f'-N 1 --job-name={benchmarkPath.lower()}_{sst_version.replace(".", "")}_{size} {additional_srun_args}',
            sst_args     = sst_args,
            log_file     = f'{run_dir}/size_{size}',
            config_path  = f'{baseDir}/benchmarks/{benchmarkPath}/sstsimulator.conf',
            safe_display = runDisplay)

cd(f"{baseDir}")

## Watch squeue

In [ ]:
watch_queue_widget()

## Inspect results

In [ ]:
inspect_logs(f'{baseDir}/runs')

# Preprocess

In [ ]:
import os, glob
import pandas as pd

run_root_dir = f"{baseDir}/runs"
print(f'\n===== running extract under {run_root_dir} =====')

combined_frames = []

for version_dir in sorted(glob.glob(f"{run_root_dir}/sst_*")):
    if not os.path.isdir(version_dir):
        continue

    sst_version = os.path.basename(version_dir).replace('sst_', '').replace('_', '.')
    log_files = sorted(glob.glob(f"{version_dir}/size_*"))

    if len(log_files) == 0:
        print(f'No logs found under {version_dir}, skipping.')
        continue

    cd(version_dir)
    data = extract_sst_output_in_files(log_files)
    csv_lines = convert_to_csv(data)
    version_csv = f"{version_dir}/results.csv"

    with open(version_csv, 'w') as f:
        f.write('\n'.join(csv_lines))

    if os.path.exists(version_csv):
        version_df = pd.read_csv(version_csv)
        version_df['SST Version'] = sst_version
        combined_frames.append(version_df)
    else:
        print(f'WARNING: {version_csv} not created')

csv_name = f"{run_root_dir}/results.csv"

if len(combined_frames) == 0:
    print('No version data found; combined CSV not created.')
else:
    combined_df = pd.concat(combined_frames, ignore_index=True)
    combined_df.to_csv(csv_name, index=False)
    print(f'\n===== {csv_name} =====')
    print(combined_df.to_csv(index=False))

cd(baseDir)

# Plot

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

try:
    df = pd.read_csv(f"{baseDir}/runs/results.csv")
except FileNotFoundError as e:
    print(f'ERROR: File not found - {e.filename}')
    raise StopExecution()

required_columns = {'Size', 'SST Version', 'total_duration'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Missing expected columns in results.csv: {sorted(missing_columns)}')

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

plot_value = 'total_duration'
ylabel = 'Total duration (secs)'

# Plot each SST version on the same axes for direct comparison.
for sst_version, group in df.groupby('SST Version'):
    group = group.sort_values('Size')
    ax.plot(group['Size'], group[plot_value], marker='o', linewidth=2, label=f'SST {sst_version}')

ax.set_title(f'SST {benchmarkPath.upper()} single-node component scaling ({plot_value})')
ax.set_xlabel('Number of components')
ax.set_ylabel(ylabel)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()